# Full fine-tune with class weights

Same as notebook 04 with square-root inverse-frequency class weights, to address the rare OBJECTIVE class. T4 GPU, about 15 minutes.

In [ ]:
# ===== STEP 4b: the same BiomedBERT fine-tune, with CLASS WEIGHTS to fix the rare OBJECTIVE class =====
# Runtime > Change runtime type > T4 GPU. Takes roughly 15 to 25 minutes.
import os, time, json
os.environ["WANDB_DISABLED"] = "true"; os.environ["WANDB_MODE"] = "disabled"
import numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, accuracy_score, classification_report
from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/sci-classify/data"; RES = "/content/drive/MyDrive/sci-classify/results"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to a T4 runtime")
LABELS = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]; L2I = {l: i for i, l in enumerate(LABELS)}
train = pd.read_parquet(f"{DATA}/train.parquet"); dev = pd.read_parquet(f"{DATA}/dev.parquet")
test = pd.read_parquet(f"{DATA}/test.parquet"); test_llm = pd.read_parquet(f"{DATA}/test_llm_1000.parquet")

# Same position clue as before, written into the text: "[3 of 9] sentence..."
pos = lambda df: [f"[{l + 1} of {t}] {s}" for s, l, t in zip(df.text, df.line_number, df.total_lines)]

MODEL_IDS = ["microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext", "dmis-lab/biobert-base-cased-v1.2"]
for mid in MODEL_IDS:
    try:
        tok = AutoTokenizer.from_pretrained(mid)
        model = AutoModelForSequenceClassification.from_pretrained(
            mid, num_labels=5, id2label=dict(enumerate(LABELS)), label2id=L2I); used = mid; break
    except Exception as e:
        print("Could not load", mid, "->", e)
print("Using", used)

def to_ds(df):
    ds = Dataset.from_dict({"text": pos(df), "label": [L2I[l] for l in df.label]})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True, remove_columns=["text"])

ds_train = to_ds(train)                                  # all 180,040 sentences
ds_dev = to_ds(dev.sample(5000, random_state=0))         # a dev sample to monitor training

def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"), "accuracy": accuracy_score(p.label_ids, preds)}

total_steps = -(-len(ds_train) // 64)                    # one epoch at batch size 64
args = TrainingArguments(output_dir="/content/biomedbert_rct_weighted", num_train_epochs=1, learning_rate=3e-5,
                         per_device_train_batch_size=64, per_device_eval_batch_size=256,
                         warmup_steps=int(0.06 * total_steps),    # 6% warm-up (newer transformers dropped warmup_ratio)
                         weight_decay=0.01, fp16=torch.cuda.is_available(), eval_strategy="steps", eval_steps=700,
                         logging_steps=100, save_strategy="no", report_to="none", seed=42)
# Class weights: rarer classes count more in the loss. Square-root of inverse frequency softens the
# correction, so the common classes are not sacrificed (full inverse weighting often over-corrects).
freq = train.label.value_counts(normalize=True).reindex(LABELS).values
weights = torch.tensor((1 / freq) ** 0.5 / ((1 / freq) ** 0.5).mean(), dtype=torch.float)
print("Class weights:", dict(zip(LABELS, weights.round(decimals=2).tolist())))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_dev,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
t0 = time.time(); trainer.train(); train_secs = time.time() - t0

def predict(df):
    t1 = time.time(); out = trainer.predict(to_ds(df)); secs = time.time() - t1
    return [LABELS[i] for i in out.predictions.argmax(-1)], 1000 * secs / len(df)

def bootstrap_ci(y, p, n=1000, seed=0):
    rng = np.random.default_rng(seed); y, p = np.asarray(y), np.asarray(p); idx = np.arange(len(y))
    s = [f1_score(y[i], p[i], average="macro") for i in (rng.choice(idx, len(idx)) for _ in range(n))]
    return np.percentile(s, [2.5, 97.5])

p_full, ms = predict(test); p_llm, _ = predict(test_llm)
lo, hi = bootstrap_ci(test.label, p_full); lo2, hi2 = bootstrap_ci(test_llm.label, p_llm)
name = f"Fine-tuned {used.split('/')[-1]} + position + class weights (180k labels)"
print(f"\n{name}")
print(f"  full test : macro-F1 {f1_score(test.label, p_full, average='macro'):.3f} ({lo:.3f} to {hi:.3f}), "
      f"accuracy {accuracy_score(test.label, p_full):.3f}")
print(f"  1000 set  : macro-F1 {f1_score(test_llm.label, p_llm, average='macro'):.3f} ({lo2:.3f} to {hi2:.3f})")
print(f"  training {train_secs/60:.1f} min, prediction {ms:.2f} ms/sentence on a T4")
print("\nPer-class F1 (full test):"); print(classification_report(test.label, p_full, labels=LABELS, digits=3))

json.dump({"model": name, "macro_f1_full": f1_score(test.label, p_full, average="macro"), "ci_full": [lo, hi],
           "macro_f1_1000": f1_score(test_llm.label, p_llm, average="macro"), "ci_1000": [lo2, hi2],
           "train_minutes": train_secs / 60, "ms_per_sentence": ms, "preds_1000": p_llm},
          open(f"{RES}/step4b_finetune_weighted.json", "w"))
prev = json.load(open(f"{RES}/step4_finetune.json"))
p_prev = prev["preds_1000"]
print("OBJECTIVE F1 on the 1,000 set: unweighted "
      f"{f1_score(test_llm.label, p_prev, labels=['OBJECTIVE'], average='macro'):.3f} -> weighted "
      f"{f1_score(test_llm.label, p_llm, labels=['OBJECTIVE'], average='macro'):.3f}")
print("Saved to", RES)
